# STASH-Search: Pipeline Visualization on Kaggle (2x T4 16GB Edition)
### Architecture: Multi-GPU (GPU 0 + GPU 1 + CPU Offload)
This notebook runs the **STASH-Search** pipeline on STASH-Bench and produces intermediate visual telemetry and qualitative overlays.

#### Kaggle Hardware Configuration:
- **Accelerators**: 2x NVIDIA Tesla T4 (16GB VRAM each, Turing `sm_75`).
- **Model Split**:
  - **GPU 0** (Capped at 12 GiB): Holds input vision embeddings, initial transformer layers, and the **SigLIP Dual-Encoder** (`google/siglip2-so400m-patch16-512`, ~1.8GB).
  - **GPU 1** (Capped at 14 GiB): Holds subsequent transformer layers.
  - **CPU RAM** (Capped at 20 GiB): System RAM offload buffer (Kaggle provides ~30GB CPU RAM) to prevent CUDA OOM.
- **Precision**: `torch.float16` (native Tensor Core acceleration on Turing `sm_75`).
- **Visual Artifacts**:
  - `pipeline_overview.jpg`: Multi-panel overview of Query, Gallery, Quadtree, Screening, and Winning Crop.
  - `stage2_quadtree.jpg`: Quadtree decomposition with SigLIP score badges.
  - `stage3_screening.jpg`: Screened, fused, and expanded candidate crops.
  - `stage4_evidence.jpg`: Winning evidence crop highlighted in emerald green.
  - `crops/`: Extracted native-resolution crops saved at 100% quality.
  - `pipeline_trace.json`: Full machine-readable telemetry trace.


In [ ]:
# Cell 1: Dual-GPU & System RAM Sanity Check
import os
import sys
import torch

print("=" * 74)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count()
print(f"GPU Count       : {gpu_count}")
for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    total_gb = props.total_memory / (1024**3)
    print(f"  • GPU {i}: {props.name} ({total_gb:.2f} GB VRAM, sm_{props.major}{props.minor})")
print("=" * 74)

if gpu_count < 2:
    print("[Notice] Kaggle kernel has < 2 GPUs visible. To enable 2x T4 on Kaggle:")
    print("         Go to the right-hand Settings pane -> Accelerator -> Select 'GPU T4 x2'.")

!nvidia-smi


In [ ]:
# Cell 2: Clone / Update MSSearch Repository (Branch: minimal/quali_kaggle)
import os
import sys

REPO_DIR = "/kaggle/working/MSSearch"
BRANCH = "minimal/quali_kaggle"
REPO_URL = "https://github.com/tothanhdat2006/MSSearch.git"

# Support Kaggle Secrets for GitHub Token if repository is private
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    gh_token = user_secrets.get_secret("GITHUB_TOKEN") or user_secrets.get_secret("GH_TOKEN")
    if gh_token:
        REPO_URL = f"https://{gh_token}@github.com/tothanhdat2006/MSSearch.git"
        print("[Auth] Configured GitHub Token from Kaggle Secrets.")
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("[Auth] Configured HuggingFace Token from Kaggle Secrets.")
except Exception:
    pass

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (branch: {BRANCH}) -> {REPO_DIR}...")
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"Updating repository in {REPO_DIR}...")
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


In [ ]:
# Cell 3: Install Dependencies
!pip install --prefer-binary --upgrade-strategy only-if-needed \
    "transformers>=4.45.0" \
    "accelerate>=0.26.0" \
    "torchvision" \
    "pillow" \
    "spacy" \
    "timm" \
    "matplotlib" \
    "sentencepiece"

# Download spaCy English model for NLP target entity extraction
!python -m spacy download en_core_web_sm --no-deps


In [ ]:
# Cell 4: Path Configuration & Dataset Detection
import os
import json
from PIL import Image

# 1. Search for STASHBench dataset under Kaggle inputs
KAGGLE_INPUT_DIR = "/kaggle/input"
CANDIDATE_DATASET_PATHS = [
    "/kaggle/input/stashbench/annotations_part1234_merged.json",
    "/kaggle/input/stash-bench/annotations_part1234_merged.json",
    "/kaggle/input/multisource-search/STASHBench/annotations_part1234_merged.json",
]
CANDIDATE_IMAGE_DIRS = [
    "/kaggle/input/stashbench/pool",
    "/kaggle/input/stash-bench/pool",
    "/kaggle/input/multisource-search/STASHBench/pool",
    "/kaggle/input/stashbench",
]

DATASET_PATH = None
IMAGE_DIR = None

for p in CANDIDATE_DATASET_PATHS:
    if os.path.isfile(p):
        DATASET_PATH = p
        break

for d in CANDIDATE_IMAGE_DIRS:
    if os.path.isdir(d):
        IMAGE_DIR = d
        break

# Working output paths
OUTPUT_DIR = "/kaggle/working/results"
VIS_DIR = "/kaggle/working/vis/stash"
OFFLOAD_DIR = "/tmp/offload"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)
os.makedirs(OFFLOAD_DIR, exist_ok=True)

print(f"Dataset Path : {DATASET_PATH} (found: {DATASET_PATH is not None})")
print(f"Image Dir    : {IMAGE_DIR} (found: {IMAGE_DIR is not None})")
print(f"Vis Output   : {VIS_DIR}")
print(f"Offload Dir  : {OFFLOAD_DIR}")

if not DATASET_PATH or not IMAGE_DIR:
    print("\n[Notice] STASHBench dataset not detected in /kaggle/input/.")
    print("         To attach your dataset in Kaggle: Click '+ Add Input' -> Upload/Select STASHBench.")
    print("         Generating demo gallery in /kaggle/working/demo_stash for immediate test run...")
    
    # Generate dummy demo gallery for verification
    demo_dir = "/kaggle/working/demo_stash"
    demo_pool = os.path.join(demo_dir, "pool")
    os.makedirs(demo_pool, exist_ok=True)
    
    from PIL import ImageDraw
    colors = ["#2b5c8f", "#8f2b38", "#2b8f4c", "#d4a017", "#6b3ba7"]
    for idx, c in enumerate(colors):
        im = Image.new("RGB", (1024, 768), color=c)
        draw = ImageDraw.Draw(im)
        draw.rectangle([400, 300, 624, 468], fill="white")
        draw.text((450, 370), f"SAMPLE {idx}", fill="black")
        im.save(os.path.join(demo_pool, f"demo_img_{idx:02d}.jpg"))
        
    demo_anno = [
        {
            "sample_id": "kaggle_demo_001",
            "question": "What text is written inside the white box?",
            "target": "white box",
            "image": "demo_img_02.jpg",
            "candidate_images": [f"demo_img_{i:02d}.jpg" for i in range(5)],
            "answer": "SAMPLE 2",
            "category": "demo"
        }
    ]
    DATASET_PATH = os.path.join(demo_dir, "annotations.json")
    IMAGE_DIR = demo_pool
    with open(DATASET_PATH, "w") as f:
        json.dump(demo_anno, f, indent=2)
    print(f"[Demo Ready] Created demo sample in: {DATASET_PATH}")


In [ ]:
# Cell 5: Initialize STASH-Search Engine with 2x T4 + CPU Split
import torch
from minimal.config import STASHConfig
from minimal.engine import STASHSearchEngine

# Models
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"  # Or "Qwen/Qwen2.5-VL-7B-Instruct"
SIGLIP_NAME = "google/siglip2-so400m-patch16-512"

# Multi-GPU + CPU Memory Split Architecture:
# - GPU 0: 12GiB cap (leaves 3GB for SigLIP ~1.8GB + vision input activations)
# - GPU 1: 14GiB cap (for remaining transformer layers)
# - CPU: 20GiB offload buffer in system RAM to guarantee 0% OOM probability
max_memory = {
    0: "12GiB",
    1: "14GiB",
    "cpu": "20GiB"
}

# Configuration
config = STASHConfig(
    k=20,
    saliency_plateau_gain_threshold=0.04,
    answer_dedup_similarity=0.85,
    save_vis=True,
    vis_dir=VIS_DIR,
    save_crops=True,
    verbose=True
)

print(f"Loading STASHSearchEngine...")
print(f"  • MLLM         : {MODEL_NAME}")
print(f"  • Dual-Encoder : {SIGLIP_NAME} on cuda:0 (FP16)")
print(f"  • Memory Split : {max_memory}")
print(f"  • Offload Dir  : {OFFLOAD_DIR}")

engine = STASHSearchEngine.from_pretrained(
    mllm_name_or_path=MODEL_NAME,
    dual_encoder_name=SIGLIP_NAME,
    config=config,
    device="cuda:0",
    nlp_model_path="en_core_web_sm",
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory=max_memory,
    offload_folder=OFFLOAD_DIR
)

print("\nEngine successfully loaded!")
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / (1024**3)
    reserved = torch.cuda.memory_reserved(i) / (1024**3)
    print(f"  • GPU {i} Memory: Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB")


In [ ]:
# Cell 6: Run STASH Visual Search with Intermediate Visual Logging
import os
import json
import time
from PIL import Image, ImageOps

# Load samples from annotation file
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    samples_data = json.load(f)

# Select number of samples to visualize
NUM_SAMPLES_TO_RUN = 5
POOL_MODE = "N_5"

print(f"Running visual search pipeline on {min(NUM_SAMPLES_TO_RUN, len(samples_data))} samples...\n")

results_summary = []

for idx, item in enumerate(samples_data[:NUM_SAMPLES_TO_RUN]):
    sample_id = item.get("sample_id", f"sample_{idx:03d}")
    question = item.get("question", "")
    gt_ans = item.get("answer", "")
    
    # Resolve image filenames
    cand_filenames = item.get("candidate_images", [item.get("image")])
    if POOL_MODE == "N_1" or len(cand_filenames) == 1:
        cand_filenames = [item.get("image")]
        
    pil_images = []
    for fn in cand_filenames:
        full_path = os.path.join(IMAGE_DIR, fn) if not os.path.isabs(fn) else fn
        if os.path.isfile(full_path):
            with Image.open(full_path) as im:
                pil_images.append(ImageOps.exif_transpose(im).convert("RGB"))
        else:
            print(f"[Warning] Image not found: {full_path}")
            
    if not pil_images:
        continue
        
    # Ground truth image index
    gt_img_fn = item.get("image", "")
    gt_idx = None
    if gt_img_fn in cand_filenames:
        gt_idx = cand_filenames.index(gt_img_fn)
        
    print("=" * 70)
    print(f"[{idx+1}/{min(NUM_SAMPLES_TO_RUN, len(samples_data))}] Sample: {sample_id}")
    print(f"Question : {question}")
    print(f"GT Answer: {gt_ans} | GT Image Index: {gt_idx}")
    print("=" * 70)
    
    t0 = time.perf_counter()
    res = engine.search(
        candidate_images=pil_images,
        question=question,
        target_hint=item.get("target"),
        sample_id=sample_id,
        pool_mode=POOL_MODE,
        gt_answer=gt_ans,
        gt_image_idx=gt_idx,
        save_vis=True,
        vis_dir=VIS_DIR
    )
    elapsed = time.perf_counter() - t0
    
    pred_ans = res.get("primary_answer")
    pred_img = res.get("primary_evidence_image")
    print(f"\n-> Final Prediction: '{pred_ans}' (Image: {pred_img}) in {elapsed:.2f}s")
    
    results_summary.append({
        "sample_id": sample_id,
        "question": question,
        "gt_answer": gt_ans,
        "pred_answer": pred_ans,
        "gt_image": gt_idx,
        "pred_image": pred_img,
        "elapsed_sec": round(elapsed, 2)
    })

print("\nExecution complete! Visualization artifacts saved under:", VIS_DIR)


In [ ]:
# Cell 7: Interactive In-Notebook Visualizer
import os
import json
import matplotlib.pyplot as plt
from PIL import Image

# Find all visualized sample directories
vis_pool_dir = os.path.join(VIS_DIR, POOL_MODE)
if os.path.isdir(vis_pool_dir):
    vis_samples = sorted(os.listdir(vis_pool_dir))
else:
    vis_samples = []

print(f"Found {len(vis_samples)} visualized samples in '{vis_pool_dir}'.\n")

for s_id in vis_samples:
    s_dir = os.path.join(vis_pool_dir, s_id)
    overview_path = os.path.join(s_dir, "pipeline_overview.jpg")
    trace_path = os.path.join(s_dir, "pipeline_trace.json")
    
    print("=" * 80)
    print(f"VISUALIZATION SUMMARY: {s_id}")
    print("=" * 80)
    
    # 1. Print structured trace
    if os.path.isfile(trace_path):
        with open(trace_path, "r", encoding="utf-8") as tf:
            trace = json.load(tf)
        print(f"Query Targets       : {trace.get('query_targets')}")
        print(f"Timing Breakdown    : {trace.get('timings_seconds')}")
        print(f"Consensus Clusters  : {len(trace.get('clusters', []))} cluster(s)")
        print(f"Winning Crop Index  : {trace.get('winning_crop_index')}")
        print(f"Predicted Answer    : {trace.get('predicted_answer')}")
        print(f"Ground Truth Answer : {trace.get('gt_answer')}")
    
    # 2. Display Pipeline Overview Composite
    if os.path.isfile(overview_path):
        print(f"\n[Composite Overview] {overview_path}")
        im_overview = Image.open(overview_path)
        plt.figure(figsize=(16, 12), dpi=120)
        plt.imshow(im_overview)
        plt.axis("off")
        plt.title(f"STASH-Search Overview: {s_id}", fontsize=14, pad=10)
        plt.show()
        
    # 3. Display Individual Stages Breakdown
    stage2_p = os.path.join(s_dir, "stage2_quadtree.jpg")
    stage3_p = os.path.join(s_dir, "stage3_screening.jpg")
    stage4_p = os.path.join(s_dir, "stage4_evidence.jpg")
    
    stage_imgs = []
    titles = []
    for p, title in [(stage2_p, "Stage 2: Quadtree Leaves"), 
                     (stage3_p, "Stage 3: Screened & Fused"), 
                     (stage4_p, "Stage 4: Winning Evidence")]:
        if os.path.isfile(p):
            stage_imgs.append(Image.open(p))
            titles.append(title)
            
    if stage_imgs:
        fig, axes = plt.subplots(1, len(stage_imgs), figsize=(6 * len(stage_imgs), 5), dpi=100)
        if len(stage_imgs) == 1:
            axes = [axes]
        for ax, im, title in zip(axes, stage_imgs, titles):
            ax.imshow(im)
            ax.set_title(title, fontsize=12)
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    # 4. Display Extracted Native-Resolution Crops
    crops_dir = os.path.join(s_dir, "crops")
    if os.path.isdir(crops_dir):
        crop_files = sorted(os.listdir(crops_dir))[:6] # Display first 6 crops
        if crop_files:
            print(f"\n[Extracted Native-Resolution Crops (first {len(crop_files)})]")
            fig, axes = plt.subplots(1, len(crop_files), figsize=(3.5 * len(crop_files), 3.5), dpi=100)
            if len(crop_files) == 1:
                axes = [axes]
            for ax, cf in zip(axes, crop_files):
                c_im = Image.open(os.path.join(crops_dir, cf))
                ax.imshow(c_im)
                ax.set_title(cf.replace(".jpg", ""), fontsize=9)
                ax.axis("off")
            plt.tight_layout()
            plt.show()


In [ ]:
# Cell 8: Zip & Download Visualizations from Kaggle Output
import shutil

zip_output = "/kaggle/working/stash_visualizations"
archive_format = "zip"

print(f"Archiving visualization artifacts from '{VIS_DIR}'...")
archive_path = shutil.make_archive(zip_output, archive_format, VIS_DIR)
archive_size_mb = os.path.getsize(archive_path) / (1024**2)

print("=" * 70)
print(f"Archive Created : {archive_path}")
print(f"Archive Size    : {archive_size_mb:.2f} MB")
print("=" * 70)
print("You can download this archive directly from Kaggle's right sidebar:")
print("  Output -> Data -> /kaggle/working/stash_visualizations.zip -> Download")


### Alternative: Run Batch Evaluation & Visualization via CLI Runner
You can also run batch evaluation directly via the CLI runner with the `--split_gpus` flag:
```bash
!python -m minimal.eval_runner \
    --benchmark stash \
    --pool_mode N_5 \
    --model_path "Qwen/Qwen3-VL-8B-Instruct" \
    --dual_encoder_name "google/siglip2-so400m-patch16-512" \
    --dataset_path "/kaggle/input/stashbench/annotations_part1234_merged.json" \
    --image_dir "/kaggle/input/stashbench/pool" \
    --split_gpus \
    --save_vis \
    --vis_dir "/kaggle/working/vis/stash" \
    --vis_max_samples 10 \
    --max_samples 10
```
